# Experiment: Autonomous Colab Full Training

This notebook is designed for Google Colab and is fully self-contained.

Success criteria:
- load train.csv and test_final.csv from Google Drive
- run leakage-safe holdout validation with a group-time split
- train CatBoost frequency and severity models
- optimize pricing on the holdout set
- refit on the full training set and save a ready-to-submit submission.csv

## Plan

1. Install dependencies and set runtime configuration.
2. Mount Google Drive and validate input files.
3. Aggregate raw data to policy level.
4. Fit a leakage-safe preprocessor on holdout-train only.
5. Train CatBoost-only frequency + severity models.
6. Optimize pricing with alpha/beta grid search.
7. Refit on full train and generate submission artifacts.

In [ ]:
%%capture
!pip -q install pandas numpy scikit-learn catboost optuna pyarrow

In [ ]:
TRAIN_CSV = "/content/drive/MyDrive/risk_case/data/train.csv"
TEST_CSV = "/content/drive/MyDrive/risk_case/data/test_final.csv"
OUTPUT_ROOT = "/content/drive/MyDrive/risk_case/artifacts/colab_full_runs"

USE_GPU = True
RUN_MODE = "release"  # "release" or "optuna"
SAMPLE_N = None       # e.g. 200000 for a quick smoke run
TIME_HOLDOUT_START = "2022-09-22 00:00:00"

TARGET_LR = 0.70
TARGET_BAND = (0.69, 0.71)
OPTUNA_TRIALS = 40

In [ ]:
from __future__ import annotations

import hashlib
import json
import logging
from dataclasses import asdict, dataclass, field
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import optuna
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, roc_auc_score
from sklearn.model_selection import GroupKFold, KFold, train_test_split

SEED = 42
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

CONTRACT_COL = "contract_number"
TARGET_CLAIM_COL = "is_claim"
TARGET_AMOUNT_COL = "claim_amount"
TARGET_COUNT_COL = "claim_cnt"
TIME_COL = "operation_date"
PREMIUM_COL = "premium"
PREMIUM_NET_COL = "premium_wo_term"

LOGGER = logging.getLogger("colab_full_training")
if not LOGGER.handlers:
    logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")

RELEASE_PARAMS = {
    "random_state": 42,
    "thread_count": 4,
    "iterations": 250,
    "learning_rate": 0.05,
    "depth": 6,
    "l2_leaf_reg": 3.0,
    "random_strength": 1.0,
    "bagging_temperature": 0.0,
    "border_count": 254,
    "reg_iterations": 477,
    "reg_learning_rate": 0.022701730462062378,
    "reg_depth": 6,
    "reg_l2_leaf_reg": 2.0503012803884313,
    "reg_random_strength": 0.7910720924659107,
    "reg_bagging_temperature": 0.38578306115544414,
    "reg_border_count": 112,
    "severity_loss_function": "TWEEDIE",
    "tweedie_variance_power": 1.3429970066671788,
}

ALPHA_GRID = np.linspace(-0.6, 2.0, 80)
BETA_GRID = np.linspace(0.8, 1.8, 80)

def json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    return str(value)

def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2, default=json_default),
        encoding="utf-8",
    )

print("Imports ready.")
print("Default run mode:", RUN_MODE)
print("Release params use CatBoost-only frequency + Tweedie severity.")

In [ ]:
def ensure_drive_mount_if_needed(*paths_to_check: str) -> None:
    needs_colab_drive = any(str(item).startswith("/content/drive") for item in paths_to_check)
    if not needs_colab_drive:
        return

    try:
        from google.colab import drive
    except Exception as exc:
        raise RuntimeError(
            "Google Drive paths were requested, but this runtime is not Google Colab."
        ) from exc

    mount_point = Path("/content/drive")
    mount_point.mkdir(parents=True, exist_ok=True)
    drive.mount(str(mount_point), force_remount=False)

RUN_MODE = str(RUN_MODE).strip().lower()
if RUN_MODE not in {"release", "optuna"}:
    raise ValueError("RUN_MODE must be either 'release' or 'optuna'.")

ensure_drive_mount_if_needed(TRAIN_CSV, TEST_CSV, OUTPUT_ROOT)

TRAIN_CSV = Path(TRAIN_CSV)
TEST_CSV = Path(TEST_CSV)
OUTPUT_ROOT = Path(OUTPUT_ROOT)

for csv_path in [TRAIN_CSV, TEST_CSV]:
    if not csv_path.exists():
        raise FileNotFoundError(f"Required file was not found: {csv_path}")

RUN_ID = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = OUTPUT_ROOT / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_CSV :", TRAIN_CSV)
print("TEST_CSV  :", TEST_CSV)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("USE_GPU   :", USE_GPU)
print("RUN_MODE  :", RUN_MODE)
print("SAMPLE_N  :", SAMPLE_N)

In [ ]:
IDENTIFIER_DROP_COLUMNS = {"unique_id", "driver_iin", "insurer_iin", "car_number"}
TARGET_AND_FINANCIAL_COLUMNS = {
    PREMIUM_COL,
    PREMIUM_NET_COL,
    TARGET_CLAIM_COL,
    TARGET_AMOUNT_COL,
    TARGET_COUNT_COL,
}

INTERACTION_REQUIRED_SOURCES = {
    "premium_per_driver": {"premium", "driver_count"},
    "premium_wo_term_per_driver": {"premium_wo_term", "driver_count"},
    "premium_per_power": {"premium", "engine_power"},
    "premium_wo_term_per_power": {"premium_wo_term", "engine_power"},
    "car_age_x_bonus_malus": {"car_age", "bonus_malus"},
    "region_x_vehicle_type": {"region_name", "vehicle_type_name"},
}

@dataclass
class PreprocessingConfig:
    grain: str = CONTRACT_COL
    add_missing_flags: bool = True
    add_missing_aggregates: bool = True
    missing_aggregate_prefixes: list[str] = field(default_factory=lambda: ["SCORE_4_", "SCORE_11_", "SCORE_"])
    missing_flag_threshold: float = 0.05
    numeric_default_strategy: str = "median"
    financial_fill_value: float = 0.0
    winsorize_low: float = 0.01
    winsorize_high: float = 0.99
    rare_category_threshold: float = 0.005
    rare_category_min_count: int = 200
    date_columns: list[str] = field(default_factory=lambda: [TIME_COL])
    date_features: list[str] = field(
        default_factory=lambda: ["month", "quarter", "dayofweek", "is_month_end", "sin_month", "cos_month"]
    )
    target_encoding_enabled: bool = True
    target_encoding_columns: list[str] = field(
        default_factory=lambda: ["model", "mark", "ownerkato", "region_name", "car_year", "bonus_malus"]
    )
    target_encoding_smoothing: float = 20.0
    target_encoding_min_samples_leaf: int = 100
    target_encoding_noise_std: float = 0.0
    frequency_encoding_enabled: bool = True
    frequency_encoding_columns: list[str] = field(
        default_factory=lambda: ["model", "mark", "ownerkato", "region_name", "car_year", "bonus_malus"]
    )
    interaction_features_enabled: bool = True
    interaction_features: list[str] = field(
        default_factory=lambda: [
            "premium_per_driver",
            "premium_wo_term_per_driver",
            "premium_per_power",
            "premium_wo_term_per_power",
            "car_age_x_bonus_malus",
            "region_x_vehicle_type",
        ]
    )
    log1p_columns: list[str] = field(
        default_factory=lambda: [PREMIUM_COL, PREMIUM_NET_COL, "engine_volume_mean", "engine_power_mean"]
    )
    drop_columns: list[str] = field(default_factory=lambda: ["unique_id", "driver_iin", "insurer_iin", "car_number"])
    target_columns: list[str] = field(default_factory=lambda: [TARGET_CLAIM_COL, TARGET_AMOUNT_COL, TARGET_COUNT_COL])
    force_keep_features: list[str] = field(default_factory=lambda: [PREMIUM_COL, PREMIUM_NET_COL])
    progress_log_every_n: int = 25
    feature_pruning_enabled: bool = True
    feature_pruning_drop_exact_duplicates: bool = True
    feature_pruning_drop_missing_share: bool = True
    feature_pruning_corr_threshold: float = 0.995
    drift_pruning_enabled: bool = True
    drift_pruning_time_column: str = TIME_COL
    drift_pruning_reference_share: float = 0.7
    drift_pruning_psi_threshold: float = 0.6
    drift_pruning_bins: int = 10
    drift_pruning_min_rows: int = 500
    drift_pruning_exclude_columns: list[str] = field(default_factory=list)
    drift_pruning_exclude_patterns: list[str] = field(default_factory=lambda: ["operation_date_*"])

@dataclass
class FittedPreprocessor:
    config: PreprocessingConfig
    numeric_columns: list[str]
    categorical_columns: list[str]
    date_input_columns: list[str]
    date_feature_columns: list[str]
    categorical_output_columns: list[str]
    numeric_fill_values: dict[str, float]
    winsor_bounds: dict[str, tuple[float, float]]
    missing_flag_columns: list[str]
    missing_aggregate_definitions: dict[str, list[str]]
    rare_category_kept: dict[str, list[str]]
    target_encoding_maps: dict[str, dict[str, float]]
    target_encoding_global_mean: float
    frequency_encoding_maps: dict[str, dict[str, float]]
    interaction_feature_columns: list[str]
    feature_columns: list[str]
    feature_pruning_report: dict[str, Any] = field(default_factory=dict)

    def to_dict(self) -> dict[str, Any]:
        payload = asdict(self)
        payload["winsor_bounds"] = {
            key: [float(bounds[0]), float(bounds[1])]
            for key, bounds in self.winsor_bounds.items()
        }
        return payload

PREPROCESSING_CONFIG = PreprocessingConfig()

def unique_in_order(items: list[str]) -> list[str]:
    seen: set[str] = set()
    ordered: list[str] = []
    for item in items:
        if item in seen:
            continue
        seen.add(item)
        ordered.append(item)
    return ordered

def aggregate_to_policy_level(df: pd.DataFrame, contract_col: str = CONTRACT_COL) -> pd.DataFrame:
    if contract_col not in df.columns:
        raise ValueError(f"Missing required contract column: {contract_col}")

    grouped = df.groupby(contract_col, dropna=False)
    aggregation_map: dict[str, Any] = {}
    rename_map: dict[str, str] = {}

    for column in df.columns:
        if column == contract_col:
            continue
        if column in IDENTIFIER_DROP_COLUMNS:
            continue

        series = df[column]
        if column in TARGET_AND_FINANCIAL_COLUMNS:
            aggregation_map[column] = "max"
            continue

        if pd.api.types.is_numeric_dtype(series):
            aggregation_map[column] = "mean"
            if not column.startswith("SCORE_"):
                rename_map[column] = f"{column}_mean"
        else:
            aggregation_map[column] = "first"

    policy_df = grouped.agg(aggregation_map).reset_index()
    if rename_map:
        policy_df = policy_df.rename(columns=rename_map)

    driver_count = grouped.size().reset_index(name="driver_count")
    policy_df = policy_df.merge(driver_count, on=contract_col, how="left")

    for target_col in [TARGET_CLAIM_COL, TARGET_COUNT_COL, TARGET_AMOUNT_COL]:
        if target_col in policy_df.columns:
            policy_df[target_col] = pd.to_numeric(policy_df[target_col], errors="coerce").fillna(0.0)
    if TARGET_CLAIM_COL in policy_df.columns:
        policy_df[TARGET_CLAIM_COL] = policy_df[TARGET_CLAIM_COL].astype(int)

    return policy_df

def split_by_group(
    policy_df: pd.DataFrame,
    group_column: str = CONTRACT_COL,
    test_size: float = 0.2,
    random_state: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    if group_column not in policy_df.columns:
        raise ValueError(f"Group split requires column: {group_column}")

    groups = policy_df[group_column].astype("string").fillna("__missing_group__").astype(str)
    unique_groups = pd.Series(groups.unique(), dtype="string")
    if len(unique_groups) < 2:
        raise ValueError("Group split requires at least two unique groups")

    train_groups, valid_groups = train_test_split(
        unique_groups,
        test_size=test_size,
        random_state=random_state,
        shuffle=True,
    )
    train_df = policy_df.loc[groups.isin(train_groups.astype(str))].copy()
    valid_df = policy_df.loc[groups.isin(valid_groups.astype(str))].copy()
    if train_df.empty or valid_df.empty:
        raise ValueError("Group split produced an empty train or valid partition")
    return train_df, valid_df

def split_policy_train_valid(
    policy_df: pd.DataFrame,
    time_holdout_start: str | None,
    group_column: str = CONTRACT_COL,
    time_column: str = TIME_COL,
    test_size: float = 0.2,
    random_state: int = SEED,
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, Any]]:
    if time_column in policy_df.columns:
        parsed_time = pd.to_datetime(policy_df[time_column], errors="coerce")
        non_null_time = parsed_time.dropna()
        cutoff = pd.to_datetime(time_holdout_start, errors="coerce") if time_holdout_start else None

        if cutoff is None and not non_null_time.empty:
            quantile = float(max(0.5, min(0.95, 1.0 - test_size)))
            cutoff = pd.Timestamp(non_null_time.quantile(quantile))

        if cutoff is not None and not pd.isna(cutoff):
            valid_mask = parsed_time >= cutoff
            train_df = policy_df.loc[~valid_mask].copy()
            valid_df = policy_df.loc[valid_mask].copy()

            if group_column in policy_df.columns and not train_df.empty and not valid_df.empty:
                train_groups = set(train_df[group_column].astype(str))
                overlap_mask = valid_df[group_column].astype(str).isin(train_groups)
                if bool(overlap_mask.any()):
                    valid_df = valid_df.loc[~overlap_mask].copy()

            if not train_df.empty and not valid_df.empty:
                split_meta = {
                    "scheme": "group_time",
                    "group_column": group_column,
                    "time_column": time_column,
                    "time_holdout_start": str(cutoff),
                    "train_rows": int(len(train_df)),
                    "valid_rows": int(len(valid_df)),
                }
                return train_df, valid_df, split_meta

    train_df, valid_df = split_by_group(
        policy_df=policy_df,
        group_column=group_column,
        test_size=test_size,
        random_state=random_state,
    )
    split_meta = {
        "scheme": "group",
        "group_column": group_column,
        "time_column": time_column,
        "time_holdout_start": None,
        "train_rows": int(len(train_df)),
        "valid_rows": int(len(valid_df)),
    }
    return train_df, valid_df, split_meta

def _protect_columns(config: PreprocessingConfig) -> set[str]:
    return {config.grain, *config.target_columns}

def _normalize_series_to_category(series: pd.Series, kept: list[str] | None) -> pd.Series:
    normalized = series.astype("string").fillna("missing").astype(str)
    kept_set = set(kept or [])
    if kept_set:
        normalized = normalized.where(normalized.isin(kept_set), "other")
    return normalized

def _choose_fill_value(series: pd.Series, config: PreprocessingConfig, column: str) -> float:
    if column in {PREMIUM_COL, PREMIUM_NET_COL, TARGET_AMOUNT_COL, TARGET_COUNT_COL}:
        return float(config.financial_fill_value)

    numeric = pd.to_numeric(series, errors="coerce")
    if config.numeric_default_strategy == "zero":
        return 0.0

    median = numeric.median()
    return 0.0 if pd.isna(median) else float(median)

def _clip_quantiles(values: pd.Series, low: float, high: float) -> tuple[float, float]:
    if values.dropna().empty:
        return 0.0, 0.0
    q_low = float(values.quantile(low))
    q_high = float(values.quantile(high))
    if q_low > q_high:
        q_low, q_high = q_high, q_low
    return q_low, q_high

def _canonicalize_numeric_like_string(series: pd.Series, column: str | None = None) -> pd.Series:
    cleaned = series.astype("string")
    has_separator = cleaned.str.contains(r"[\u00a0\s]", regex=True, na=False)
    cleaned = cleaned.str.replace("\u00a0", "", regex=False)
    cleaned = cleaned.str.replace(" ", "", regex=False)
    cleaned = cleaned.str.strip()

    if column == "car_year":
        compact_year_mask = has_separator & cleaned.str.fullmatch(r"[12]\d{2}", na=False)
        if bool(compact_year_mask.any()):
            first = cleaned.str.slice(0, 1)
            suffix = cleaned.str.slice(1, 3)
            century_digit = first.map({"1": "9", "2": "0"}).fillna("")
            expanded = first + century_digit + suffix
            cleaned = cleaned.where(~compact_year_mask, expanded)

    lowered = cleaned.str.lower()
    cleaned = cleaned.mask(lowered.isin({"", "nan", "none", "null", "na", "n/a"}), pd.NA)
    return cleaned

def _clean_numeric_like_column(
    series: pd.Series,
    column: str,
    keep_numeric_dtype: bool,
) -> tuple[pd.Series, dict[str, int]]:
    non_null_mask = series.notna()
    raw_numeric = pd.to_numeric(series, errors="coerce")
    non_numeric_before = int((non_null_mask & raw_numeric.isna()).sum())

    canonical = _canonicalize_numeric_like_string(series, column=column)
    cleaned_numeric = pd.to_numeric(canonical, errors="coerce")
    rescued = int((non_null_mask & raw_numeric.isna() & cleaned_numeric.notna()).sum())
    coerced_to_missing = int((non_null_mask & cleaned_numeric.isna()).sum())

    if keep_numeric_dtype:
        cleaned = cleaned_numeric.astype(float)
    else:
        cleaned = pd.Series(pd.NA, index=series.index, dtype="string")
        valid_mask = cleaned_numeric.notna()
        if bool(valid_mask.any()):
            valid_numeric = cleaned_numeric.loc[valid_mask]
            rounded = np.round(valid_numeric.to_numpy())
            int_like = pd.Series(
                np.isclose(valid_numeric.to_numpy(), rounded, rtol=0.0, atol=1e-9),
                index=valid_numeric.index,
            )
            if bool(int_like.any()):
                int_values = valid_numeric.loc[int_like].round().astype("Int64").astype("string")
                cleaned.loc[int_values.index] = int_values
            float_like = ~int_like
            if bool(float_like.any()):
                float_values = valid_numeric.loc[float_like].map(lambda value: format(float(value), "g"))
                cleaned.loc[float_values.index] = float_values.astype("string")

    return cleaned, {
        "non_null": int(non_null_mask.sum()),
        "non_numeric_before": non_numeric_before,
        "rescued": rescued,
        "coerced_to_missing": coerced_to_missing,
    }

def _normalize_numeric_like_columns(df: pd.DataFrame, stage: str) -> pd.DataFrame:
    for column in ("bonus_malus", "car_year"):
        if column not in df.columns:
            continue
        keep_numeric_dtype = bool(pd.api.types.is_numeric_dtype(df[column]))
        cleaned, stats = _clean_numeric_like_column(df[column], column=column, keep_numeric_dtype=keep_numeric_dtype)
        df[column] = cleaned
        if stats["non_numeric_before"] > 0 or stats["rescued"] > 0:
            LOGGER.info(
                "Numeric-like cleanup[%s]: column=%s rescued=%d coerced_to_missing=%d",
                stage,
                column,
                stats["rescued"],
                stats["coerced_to_missing"],
            )
    return df

def _prefix_token(prefix: str) -> str:
    cleaned = prefix.lower().strip("_")
    cleaned = "".join(ch if ch.isalnum() or ch == "_" else "_" for ch in cleaned)
    while "__" in cleaned:
        cleaned = cleaned.replace("__", "_")
    return cleaned or "missing"

def _fit_target_encoding_map(
    category: pd.Series,
    target: pd.Series,
    global_mean: float,
    smoothing: float,
    min_samples_leaf: int,
) -> dict[str, float]:
    frame = pd.DataFrame({
        "category": category.astype(str),
        "target": pd.to_numeric(target, errors="coerce").fillna(0.0),
    })
    grouped = frame.groupby("category", dropna=False)["target"].agg(["mean", "count"])
    counts = grouped["count"].astype(float)
    means = grouped["mean"].astype(float)
    smoothing_term = np.full(len(counts), max(smoothing, 0.0), dtype=float)
    if min_samples_leaf > 0:
        smoothing_term = np.where(counts < min_samples_leaf, smoothing_term * 2.0, smoothing_term)
    encoded = (means * counts + global_mean * smoothing_term) / np.where(
        counts + smoothing_term == 0, 1.0, counts + smoothing_term
    )
    return {str(key): float(value) for key, value in encoded.to_dict().items()}

def _fit_frequency_encoding_map(category: pd.Series) -> dict[str, float]:
    freq = category.astype(str).value_counts(dropna=False, normalize=True)
    return {str(key): float(value) for key, value in freq.to_dict().items()}

def _build_date_features(series: pd.Series, prefix: str, features: list[str]) -> tuple[dict[str, pd.Series], list[str]]:
    dt = pd.to_datetime(series, errors="coerce")
    generated: dict[str, pd.Series] = {}
    names: list[str] = []

    for feature_name in features:
        col_name = f"{prefix}_{feature_name}"
        if feature_name == "month":
            generated[col_name] = dt.dt.month.fillna(0).astype(float)
        elif feature_name == "quarter":
            generated[col_name] = dt.dt.quarter.fillna(0).astype(float)
        elif feature_name == "dayofweek":
            generated[col_name] = dt.dt.dayofweek.fillna(0).astype(float)
        elif feature_name == "is_month_end":
            generated[col_name] = dt.dt.is_month_end.fillna(False).astype(int)
        elif feature_name == "sin_month":
            month = dt.dt.month.fillna(0).astype(float)
            generated[col_name] = np.sin(2.0 * np.pi * month / 12.0)
        elif feature_name == "cos_month":
            month = dt.dt.month.fillna(0).astype(float)
            generated[col_name] = np.cos(2.0 * np.pi * month / 12.0)
        else:
            continue
        names.append(col_name)

    return generated, names

def _safe_numeric_series(df: pd.DataFrame, candidates: list[str]) -> pd.Series | None:
    for candidate in candidates:
        if candidate in df.columns:
            return pd.to_numeric(df[candidate], errors="coerce")
    return None

def _safe_divide_series(numerator: pd.Series, denominator: pd.Series, fallback: float = 1.0) -> pd.Series:
    denominator_values = pd.to_numeric(denominator, errors="coerce").fillna(0.0)
    denominator_safe = np.where(np.abs(denominator_values.to_numpy(dtype=float)) <= 1e-9, fallback, denominator_values)
    result = pd.to_numeric(numerator, errors="coerce").fillna(0.0).to_numpy(dtype=float) / denominator_safe
    return pd.Series(result, index=numerator.index, dtype=float)

def _compute_score_statistics(df: pd.DataFrame) -> dict[str, pd.Series | None]:
    score_columns = [
        col
        for col in df.columns
        if col.startswith("SCORE_")
        and col not in {TARGET_CLAIM_COL, TARGET_AMOUNT_COL, TARGET_COUNT_COL}
        and pd.api.types.is_numeric_dtype(df[col])
    ]
    if not score_columns:
        return {"score_mean": None, "score_std": None, "score_median": None}
    score_frame = df[score_columns].apply(pd.to_numeric, errors="coerce")
    return {
        "score_mean": score_frame.mean(axis=1, skipna=True).fillna(0.0),
        "score_std": score_frame.std(axis=1, ddof=0, skipna=True).fillna(0.0),
        "score_median": score_frame.median(axis=1, skipna=True).fillna(0.0),
    }

def _build_interaction_sources(df: pd.DataFrame) -> dict[str, pd.Series | None]:
    score_stats = _compute_score_statistics(df)
    return {
        "premium": _safe_numeric_series(df, [PREMIUM_COL]),
        "premium_wo_term": _safe_numeric_series(df, [PREMIUM_NET_COL]),
        "driver_count": _safe_numeric_series(df, ["driver_count"]),
        "engine_power": _safe_numeric_series(df, ["engine_power_mean", "engine_power"]),
        "car_age": _safe_numeric_series(df, ["car_age_mean", "car_age"]),
        "bonus_malus": _safe_numeric_series(df, ["bonus_malus_mean", "bonus_malus"]),
        "score_mean": score_stats["score_mean"],
        "score_std": score_stats["score_std"],
        "score_median": score_stats["score_median"],
    }

def _compute_interaction_feature_series(
    *,
    feature_name: str,
    df: pd.DataFrame,
    sources: dict[str, pd.Series | None],
) -> pd.Series | None:
    premium = sources.get("premium")
    premium_net = sources.get("premium_wo_term")
    driver_count = sources.get("driver_count")
    engine_power = sources.get("engine_power")
    car_age = sources.get("car_age")
    bonus_malus = sources.get("bonus_malus")

    if feature_name == "premium_per_driver":
        return None if premium is None or driver_count is None else _safe_divide_series(premium, driver_count)
    if feature_name == "premium_wo_term_per_driver":
        return None if premium_net is None or driver_count is None else _safe_divide_series(premium_net, driver_count)
    if feature_name == "premium_per_power":
        return None if premium is None or engine_power is None else _safe_divide_series(premium, engine_power)
    if feature_name == "premium_wo_term_per_power":
        return None if premium_net is None or engine_power is None else _safe_divide_series(premium_net, engine_power)
    if feature_name == "car_age_x_bonus_malus":
        return None if car_age is None or bonus_malus is None else car_age.fillna(0.0) * bonus_malus.fillna(0.0)
    if feature_name == "region_x_vehicle_type":
        if "region_name" not in df.columns or "vehicle_type_name" not in df.columns:
            return None
        return (
            df["region_name"].astype("string").fillna("missing").astype(str)
            + "__"
            + df["vehicle_type_name"].astype("string").fillna("missing").astype(str)
        )
    return None

def _column_signature(series: pd.Series) -> str:
    fingerprint = pd.util.hash_pandas_object(series, index=False).values.tobytes()
    return hashlib.sha1(fingerprint).hexdigest()

def _population_stability_index(reference: pd.Series, current: pd.Series, bins: int = 10, eps: float = 1e-6) -> float | None:
    ref = pd.to_numeric(reference, errors="coerce").replace([np.inf, -np.inf], np.nan)
    cur = pd.to_numeric(current, errors="coerce").replace([np.inf, -np.inf], np.nan)
    ref_valid = ref.dropna()
    cur_valid = cur.dropna()
    if len(ref_valid) < 2 or len(cur_valid) < 2:
        return None

    bins = max(int(bins), 3)
    quantiles = np.linspace(0.0, 1.0, bins + 1)
    edges = np.quantile(ref_valid.to_numpy(dtype=float), quantiles)
    edges = np.unique(edges)

    if len(edges) < 2:
        split_point = float(np.nanmedian(ref_valid.to_numpy(dtype=float)))
        edges = np.array([-np.inf, split_point, np.inf], dtype=float)
    else:
        edges = edges.astype(float)
        edges[0] = -np.inf
        edges[-1] = np.inf

    ref_counts, _ = np.histogram(ref_valid.to_numpy(dtype=float), bins=edges)
    cur_counts, _ = np.histogram(cur_valid.to_numpy(dtype=float), bins=edges)

    ref_pct = ref_counts.astype(float) / max(float(ref_counts.sum()), 1.0)
    cur_pct = cur_counts.astype(float) / max(float(cur_counts.sum()), 1.0)

    ref_pct = np.append(ref_pct, float(ref.isna().mean()))
    cur_pct = np.append(cur_pct, float(cur.isna().mean()))

    ref_pct = np.clip(ref_pct, eps, None)
    cur_pct = np.clip(cur_pct, eps, None)
    return float(np.sum((cur_pct - ref_pct) * np.log(cur_pct / ref_pct)))

def _resolve_time_order_index(df: pd.DataFrame, config: PreprocessingConfig) -> tuple[pd.Index | None, str | None]:
    candidate_columns: list[str] = []
    if config.drift_pruning_time_column:
        candidate_columns.append(config.drift_pruning_time_column)
    candidate_columns.extend([col for col in config.date_columns if col not in candidate_columns])

    for column in candidate_columns:
        if column not in df.columns:
            continue
        parsed = pd.to_datetime(df[column], errors="coerce")
        valid = parsed.notna()
        if int(valid.sum()) < 2:
            continue
        ordered = parsed[valid].sort_values(kind="mergesort").index
        return ordered, column
    return None, None

def _prune_feature_columns_by_statistics(
    transformed_df: pd.DataFrame,
    feature_columns: list[str],
    config: PreprocessingConfig,
    source_df: pd.DataFrame | None = None,
) -> tuple[list[str], dict[str, Any]]:
    retained = [col for col in feature_columns if col in transformed_df.columns]
    protected_from_pruning = {
        col
        for col in retained
        if col in set(config.force_keep_features) or col.endswith("_missing_cnt") or col.endswith("_missing_cnt_total")
    }

    dropped_exact_duplicates: list[dict[str, Any]] = []
    dropped_high_corr: list[dict[str, Any]] = []
    dropped_manual: list[str] = []
    dropped_drift_psi: list[dict[str, Any]] = []

    numeric_columns = [col for col in retained if pd.api.types.is_numeric_dtype(transformed_df[col])]
    if config.feature_pruning_drop_exact_duplicates and numeric_columns:
        signature_to_columns: dict[str, list[str]] = {}
        for column in numeric_columns:
            signature_to_columns.setdefault(_column_signature(transformed_df[column]), []).append(column)
        for columns in signature_to_columns.values():
            if len(columns) <= 1:
                continue
            ordered = sorted(columns, key=lambda col: (0 if col in protected_from_pruning else 1, col))
            keeper = ordered[0]
            for duplicate in ordered[1:]:
                if duplicate in protected_from_pruning:
                    continue
                if duplicate in retained:
                    retained.remove(duplicate)
                    dropped_exact_duplicates.append({"dropped": duplicate, "kept": keeper})

    if config.feature_pruning_drop_missing_share and "score_missing_share" in retained:
        retained.remove("score_missing_share")
        dropped_manual.append("score_missing_share")

    corr_threshold = float(config.feature_pruning_corr_threshold)
    if 0.0 < corr_threshold < 1.0:
        corr_numeric_columns = [col for col in retained if pd.api.types.is_numeric_dtype(transformed_df[col])]
        corr_numeric_columns = sorted(corr_numeric_columns, key=lambda col: (0 if col in protected_from_pruning else 1, col))
        if len(corr_numeric_columns) >= 2:
            corr_df = transformed_df[corr_numeric_columns].apply(pd.to_numeric, errors="coerce").fillna(0.0)
            corr_matrix = corr_df.corr().abs()
            upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            to_drop: set[str] = set()
            for keeper in corr_numeric_columns:
                if keeper in to_drop:
                    continue
                high_corr = upper.index[(upper[keeper] > corr_threshold).fillna(False)].tolist()
                for duplicate in high_corr:
                    if duplicate in to_drop or duplicate in protected_from_pruning:
                        continue
                    corr_value = upper.at[duplicate, keeper]
                    to_drop.add(duplicate)
                    dropped_high_corr.append({
                        "dropped": duplicate,
                        "kept": keeper,
                        "corr": float(corr_value) if pd.notna(corr_value) else None,
                    })
            if to_drop:
                retained = [col for col in retained if col not in to_drop]

    drift_details = {
        "enabled": bool(config.drift_pruning_enabled),
        "time_column": None,
        "reference_size": 0,
        "current_size": 0,
        "psi_threshold": float(config.drift_pruning_psi_threshold),
    }

    if config.drift_pruning_enabled:
        drift_source = source_df if source_df is not None else transformed_df
        ordered_index, used_time_column = _resolve_time_order_index(drift_source, config)
        drift_details["time_column"] = used_time_column

        if ordered_index is not None:
            ordered_count = len(ordered_index)
            min_rows = max(int(config.drift_pruning_min_rows), 20)
            split_share = min(max(float(config.drift_pruning_reference_share), 0.5), 0.95)
            split_idx = int(round(ordered_count * split_share))
            split_idx = min(max(split_idx, min_rows), max(ordered_count - min_rows, 1))

            if 0 < split_idx < ordered_count:
                reference_index = ordered_index[:split_idx]
                current_index = ordered_index[split_idx:]
                drift_details["reference_size"] = int(len(reference_index))
                drift_details["current_size"] = int(len(current_index))
                exclude_columns = set(config.drift_pruning_exclude_columns)
                exclude_patterns = [str(pattern) for pattern in config.drift_pruning_exclude_patterns if str(pattern).strip()]

                for column in list(retained):
                    if column in protected_from_pruning:
                        continue
                    if column in exclude_columns or any(pd.Series([column]).str.match(pattern.replace("*", ".*"), na=False).iloc[0] for pattern in exclude_patterns):
                        continue
                    if not pd.api.types.is_numeric_dtype(transformed_df[column]):
                        continue
                    psi_value = _population_stability_index(
                        reference=transformed_df.loc[reference_index, column],
                        current=transformed_df.loc[current_index, column],
                        bins=max(int(config.drift_pruning_bins), 3),
                    )
                    if psi_value is None or psi_value <= float(config.drift_pruning_psi_threshold):
                        continue
                    retained.remove(column)
                    dropped_drift_psi.append({
                        "dropped": column,
                        "psi": float(psi_value),
                        "threshold": float(config.drift_pruning_psi_threshold),
                    })

    report = {
        "enabled": bool(config.feature_pruning_enabled or config.drift_pruning_enabled),
        "applied": bool(dropped_exact_duplicates or dropped_high_corr or dropped_manual or dropped_drift_psi),
        "before_count": len(feature_columns),
        "after_count": len(retained),
        "dropped_total": len(feature_columns) - len(retained),
        "dropped_exact_duplicates": dropped_exact_duplicates,
        "dropped_high_corr": dropped_high_corr,
        "dropped_manual": dropped_manual,
        "dropped_drift_psi": dropped_drift_psi,
        "drift": drift_details,
    }
    return retained, report

def build_oof_target_encoding_features(
    df: pd.DataFrame,
    state: FittedPreprocessor,
    target_column: str = TARGET_CLAIM_COL,
    n_splits: int = 5,
    random_state: int = SEED,
    group_column: str | None = None,
) -> pd.DataFrame:
    target_encoded_columns = [col for col in state.target_encoding_maps.keys() if col in state.categorical_columns]
    if not target_encoded_columns or target_column not in df.columns:
        return pd.DataFrame(index=df.index)

    y = pd.to_numeric(df[target_column], errors="coerce").fillna(0.0)
    if len(df) < 2:
        payload = {
            f"{col}_te": pd.Series(np.full(len(df), state.target_encoding_global_mean), index=df.index)
            for col in target_encoded_columns
        }
        return pd.DataFrame(payload, index=df.index)

    prepared_categories: dict[str, pd.Series] = {}
    for col in target_encoded_columns:
        raw_series = df[col] if col in df.columns else pd.Series("missing", index=df.index)
        prepared_categories[col] = _normalize_series_to_category(raw_series, state.rare_category_kept.get(col))

    effective_splits = max(2, int(n_splits))
    if group_column and group_column in df.columns:
        groups = df[group_column].astype("string").fillna("__missing_group__").astype(str)
        unique_groups = int(groups.nunique(dropna=False))
        if unique_groups >= 2:
            effective_splits = min(effective_splits, unique_groups)
            split_iterator = GroupKFold(n_splits=effective_splits).split(df, y, groups=groups)
        else:
            effective_splits = min(effective_splits, len(df))
            split_iterator = KFold(n_splits=effective_splits, shuffle=True, random_state=random_state).split(df)
    else:
        effective_splits = min(effective_splits, len(df))
        split_iterator = KFold(n_splits=effective_splits, shuffle=True, random_state=random_state).split(df)

    global_mean = state.target_encoding_global_mean
    oof_payload = {f"{col}_te": np.full(len(df), global_mean, dtype=float) for col in target_encoded_columns}
    index_array = np.arange(len(df))
    rng = np.random.default_rng(random_state)

    for train_idx, valid_idx in split_iterator:
        y_train = y.iloc[train_idx]
        fold_mean = float(y_train.mean()) if len(y_train) else global_mean

        for col in target_encoded_columns:
            mapping = _fit_target_encoding_map(
                category=prepared_categories[col].iloc[train_idx],
                target=y_train,
                global_mean=fold_mean,
                smoothing=state.config.target_encoding_smoothing,
                min_samples_leaf=state.config.target_encoding_min_samples_leaf,
            )
            encoded = prepared_categories[col].iloc[valid_idx].map(mapping).fillna(fold_mean).astype(float).values
            if state.config.target_encoding_noise_std > 0:
                encoded = encoded + rng.normal(0.0, state.config.target_encoding_noise_std, size=len(encoded))
            oof_payload[f"{col}_te"][index_array[valid_idx]] = encoded

    return pd.DataFrame(oof_payload, index=df.index)

def fit_preprocessor(df: pd.DataFrame, config: PreprocessingConfig) -> FittedPreprocessor:
    working = df.copy()
    protected_columns = _protect_columns(config)

    drop_candidates = [col for col in config.drop_columns if col in working.columns and col not in protected_columns]
    if drop_candidates:
        working = working.drop(columns=drop_candidates)

    working = _normalize_numeric_like_columns(working, stage="fit")
    date_input_columns = [col for col in config.date_columns if col in working.columns and col not in protected_columns]

    numeric_columns = [
        col
        for col in working.columns
        if col not in protected_columns and col not in date_input_columns and pd.api.types.is_numeric_dtype(working[col])
    ]
    categorical_columns = [
        col
        for col in working.columns
        if col not in protected_columns and col not in numeric_columns and col not in date_input_columns
    ]

    selected = set(numeric_columns + categorical_columns + date_input_columns)
    force_keep = set(config.force_keep_features + date_input_columns + config.target_encoding_columns + config.frequency_encoding_columns)
    selected |= (force_keep & selected)

    numeric_columns = [col for col in numeric_columns if col in selected]
    categorical_columns = [col for col in categorical_columns if col in selected]
    date_input_columns = [col for col in date_input_columns if col in selected]

    if not numeric_columns and PREMIUM_COL in working.columns:
        numeric_columns = [PREMIUM_COL]

    missing_flag_columns: list[str] = []
    numeric_fill_values: dict[str, float] = {}
    winsor_bounds: dict[str, tuple[float, float]] = {}

    for idx, col in enumerate(numeric_columns, start=1):
        series = pd.to_numeric(working[col], errors="coerce")
        null_rate = float(series.isna().mean())
        if config.add_missing_flags and null_rate > config.missing_flag_threshold:
            missing_flag_columns.append(col)

        fill_value = _choose_fill_value(series, config, col)
        numeric_fill_values[col] = fill_value
        filled = series.fillna(fill_value)
        winsor_bounds[col] = _clip_quantiles(filled, config.winsorize_low, config.winsorize_high)

        if idx % max(config.progress_log_every_n, 1) == 0 or idx == len(numeric_columns):
            LOGGER.info("Fit numeric progress: %d/%d", idx, len(numeric_columns))

    rare_category_kept: dict[str, list[str]] = {}
    for idx, col in enumerate(categorical_columns, start=1):
        series = working[col].astype("string").fillna("missing")
        freq = series.value_counts(dropna=False, normalize=True)
        count = series.value_counts(dropna=False)
        kept = (
            set(freq[freq >= config.rare_category_threshold].index.astype(str).tolist())
            | set(count[count >= config.rare_category_min_count].index.astype(str).tolist())
        )
        kept = sorted(kept)
        if not kept:
            kept = series.value_counts(dropna=False).head(20).index.astype(str).tolist()
        rare_category_kept[col] = kept

        if idx % max(config.progress_log_every_n, 1) == 0 or idx == len(categorical_columns):
            LOGGER.info("Fit categorical progress: %d/%d", idx, len(categorical_columns))

    date_feature_columns: list[str] = []
    for date_column in date_input_columns:
        _, names = _build_date_features(working[date_column], date_column, config.date_features)
        date_feature_columns.extend(names)

    target_series = pd.to_numeric(
        working.get(TARGET_CLAIM_COL, pd.Series(0, index=working.index)),
        errors="coerce",
    ).fillna(0.0)
    target_encoding_global_mean = float(target_series.mean()) if len(target_series) else 0.0
    target_encoding_maps: dict[str, dict[str, float]] = {}
    frequency_encoding_maps: dict[str, dict[str, float]] = {}

    if config.target_encoding_enabled:
        for column in config.target_encoding_columns:
            if column not in categorical_columns:
                continue
            prepared = _normalize_series_to_category(working[column], rare_category_kept.get(column))
            target_encoding_maps[column] = _fit_target_encoding_map(
                category=prepared,
                target=target_series,
                global_mean=target_encoding_global_mean,
                smoothing=config.target_encoding_smoothing,
                min_samples_leaf=config.target_encoding_min_samples_leaf,
            )

    if config.frequency_encoding_enabled:
        for column in config.frequency_encoding_columns:
            if column not in categorical_columns:
                continue
            prepared = _normalize_series_to_category(working[column], rare_category_kept.get(column))
            frequency_encoding_maps[column] = _fit_frequency_encoding_map(prepared)

    missing_aggregate_definitions: dict[str, list[str]] = {}
    if config.add_missing_aggregates and missing_flag_columns:
        score_flags = [f"{col}_is_missing" for col in missing_flag_columns if col.startswith("SCORE_")]
        if score_flags:
            missing_aggregate_definitions["score_missing_cnt_total"] = score_flags
            missing_aggregate_definitions["score_missing_share"] = score_flags

        for prefix in config.missing_aggregate_prefixes:
            if prefix == "SCORE_":
                continue
            prefixed_flags = [f"{col}_is_missing" for col in missing_flag_columns if col.startswith(prefix)]
            if prefixed_flags:
                missing_aggregate_definitions[f"{_prefix_token(prefix)}_missing_cnt"] = prefixed_flags

    interaction_feature_columns: list[str] = []
    if config.interaction_features_enabled:
        available_sources = _build_interaction_sources(working)
        for feature_name in config.interaction_features:
            required_sources = INTERACTION_REQUIRED_SOURCES.get(feature_name, set())
            if required_sources and not required_sources.issubset({key for key, value in available_sources.items() if value is not None} | set(working.columns)):
                continue
            interaction_feature_columns.append(feature_name)

    generated_categorical_columns = list(categorical_columns)
    if "region_x_vehicle_type" in interaction_feature_columns:
        generated_categorical_columns.append("region_x_vehicle_type")

    feature_columns = list(numeric_columns + categorical_columns + date_feature_columns)
    for col in config.log1p_columns:
        if col in feature_columns:
            feature_columns.append(f"{col}_log1p")
    for col in missing_flag_columns:
        feature_columns.append(f"{col}_is_missing")
    for column in target_encoding_maps:
        feature_columns.append(f"{column}_te")
    for column in frequency_encoding_maps:
        feature_columns.append(f"{column}_freq")
    feature_columns.extend(missing_aggregate_definitions.keys())
    feature_columns.extend(interaction_feature_columns)
    feature_columns = unique_in_order(feature_columns)

    fitted = FittedPreprocessor(
        config=config,
        numeric_columns=numeric_columns,
        categorical_columns=categorical_columns,
        date_input_columns=date_input_columns,
        date_feature_columns=unique_in_order(date_feature_columns),
        categorical_output_columns=unique_in_order(generated_categorical_columns),
        numeric_fill_values=numeric_fill_values,
        winsor_bounds=winsor_bounds,
        missing_flag_columns=missing_flag_columns,
        missing_aggregate_definitions=missing_aggregate_definitions,
        rare_category_kept=rare_category_kept,
        target_encoding_maps=target_encoding_maps,
        target_encoding_global_mean=target_encoding_global_mean,
        frequency_encoding_maps=frequency_encoding_maps,
        interaction_feature_columns=unique_in_order(interaction_feature_columns),
        feature_columns=unique_in_order(feature_columns),
        feature_pruning_report={"enabled": False, "applied": False},
    )

    if config.feature_pruning_enabled or config.drift_pruning_enabled:
        transformed_fit = transform_with_preprocessor(df, fitted)
        pruned_feature_columns, pruning_report = _prune_feature_columns_by_statistics(
            transformed_df=transformed_fit,
            feature_columns=fitted.feature_columns,
            config=config,
            source_df=df,
        )
        fitted.feature_columns = pruned_feature_columns
        fitted.feature_pruning_report = pruning_report

    return fitted

def transform_with_preprocessor(df: pd.DataFrame, state: FittedPreprocessor) -> pd.DataFrame:
    cfg = state.config
    working = df.copy()
    protected_columns = _protect_columns(cfg)

    drop_candidates = [col for col in cfg.drop_columns if col in working.columns and col not in protected_columns]
    if drop_candidates:
        working = working.drop(columns=drop_candidates)

    working = _normalize_numeric_like_columns(working, stage="transform")

    for date_column in state.date_input_columns:
        if date_column not in working.columns:
            working[date_column] = pd.NaT

    missing_flag_data: dict[str, pd.Series] = {}
    for idx, col in enumerate(state.numeric_columns, start=1):
        if col not in working.columns:
            working[col] = np.nan
        series = pd.to_numeric(working[col], errors="coerce")
        missing_mask = series.isna()

        if col in state.missing_flag_columns:
            missing_flag_data[f"{col}_is_missing"] = missing_mask.astype(int)

        fill_value = state.numeric_fill_values.get(col, 0.0)
        filled = series.fillna(fill_value)
        if col not in {PREMIUM_COL, PREMIUM_NET_COL}:
            lower, upper = state.winsor_bounds.get(col, (None, None))
            if lower is not None and upper is not None:
                filled = filled.clip(lower=lower, upper=upper)
        working[col] = filled

        if col in cfg.log1p_columns:
            working[f"{col}_log1p"] = np.log1p(filled.clip(lower=0.0))

        if idx % max(cfg.progress_log_every_n, 1) == 0 or idx == len(state.numeric_columns):
            LOGGER.info("Transform numeric progress: %d/%d", idx, len(state.numeric_columns))

    missing_flags_df = pd.DataFrame(missing_flag_data, index=working.index)
    if not missing_flags_df.empty:
        working = pd.concat([working, missing_flags_df], axis=1)

    for idx, col in enumerate(state.categorical_columns, start=1):
        if col not in working.columns:
            working[col] = "missing"
        working[col] = _normalize_series_to_category(working[col], state.rare_category_kept.get(col))

        if idx % max(cfg.progress_log_every_n, 1) == 0 or idx == len(state.categorical_columns):
            LOGGER.info("Transform categorical progress: %d/%d", idx, len(state.categorical_columns))

    for date_column in state.date_input_columns:
        generated, _ = _build_date_features(working[date_column], date_column, cfg.date_features)
        for new_col, values in generated.items():
            working[new_col] = pd.to_numeric(values, errors="coerce").fillna(0.0)

    for column, mapping in state.target_encoding_maps.items():
        if column not in working.columns:
            working[column] = "missing"
        encoded = working[column].astype("string").fillna("missing").astype(str).map(mapping)
        working[f"{column}_te"] = encoded.fillna(state.target_encoding_global_mean).astype(float)

    for column, mapping in state.frequency_encoding_maps.items():
        if column not in working.columns:
            working[column] = "missing"
        encoded = working[column].astype("string").fillna("missing").astype(str).map(mapping)
        working[f"{column}_freq"] = encoded.fillna(0.0).astype(float)

    if state.missing_aggregate_definitions:
        for output_column, source_flags in state.missing_aggregate_definitions.items():
            values = missing_flags_df.reindex(columns=source_flags, fill_value=0).sum(axis=1).astype(float)
            if output_column.endswith("_share"):
                values = values / max(len(source_flags), 1)
            working[output_column] = values

    if state.interaction_feature_columns:
        interaction_sources = _build_interaction_sources(working)
        for feature_name in state.interaction_feature_columns:
            generated = _compute_interaction_feature_series(
                feature_name=feature_name,
                df=working,
                sources=interaction_sources,
            )
            if generated is not None:
                working[feature_name] = generated

    for col in state.feature_columns:
        if col not in working.columns:
            if col in state.categorical_output_columns:
                working[col] = "missing"
            elif col.endswith("_is_missing"):
                working[col] = 0
            else:
                working[col] = 0.0

    keep_columns = [col for col in [cfg.grain, *cfg.target_columns] if col in working.columns]
    final_columns = keep_columns + [col for col in state.feature_columns if col in working.columns]

    transformed = working[final_columns].copy()
    transformed = transformed.replace([np.inf, -np.inf], np.nan)
    return transformed

In [ ]:
train_raw = pd.read_csv(TRAIN_CSV, low_memory=False)
test_raw = pd.read_csv(TEST_CSV, low_memory=False)

if SAMPLE_N is not None:
    sample_n = int(SAMPLE_N)
    if sample_n > 0 and sample_n < len(train_raw):
        train_raw = train_raw.sample(sample_n, random_state=SEED).reset_index(drop=True)

print("train_raw shape:", train_raw.shape)
print("test_raw shape :", test_raw.shape)
print("claim rate     :", float(pd.to_numeric(train_raw[TARGET_CLAIM_COL], errors="coerce").fillna(0.0).mean()))

policy_train = aggregate_to_policy_level(train_raw, contract_col=CONTRACT_COL)
policy_test = aggregate_to_policy_level(test_raw, contract_col=CONTRACT_COL)

train_policy_holdout, valid_policy, split_meta = split_policy_train_valid(
    policy_df=policy_train,
    time_holdout_start=TIME_HOLDOUT_START,
    group_column=CONTRACT_COL,
    time_column=TIME_COL,
    test_size=0.2,
    random_state=SEED,
)

holdout_preprocessor = fit_preprocessor(train_policy_holdout, PREPROCESSING_CONFIG)
train_processed = transform_with_preprocessor(train_policy_holdout, holdout_preprocessor)
valid_processed = transform_with_preprocessor(valid_policy, holdout_preprocessor)

train_oof_te = build_oof_target_encoding_features(
    df=train_policy_holdout,
    state=holdout_preprocessor,
    target_column=TARGET_CLAIM_COL,
    n_splits=5,
    random_state=SEED,
    group_column=None,
)
for column in train_oof_te.columns:
    train_processed[column] = train_oof_te[column].values

HOLDOUT_FEATURE_COLUMNS = list(holdout_preprocessor.feature_columns)
HOLDOUT_CATEGORICAL_FEATURES = [
    column for column in holdout_preprocessor.categorical_output_columns if column in HOLDOUT_FEATURE_COLUMNS
]

assert set(HOLDOUT_FEATURE_COLUMNS).issubset(train_processed.columns)
assert set(HOLDOUT_FEATURE_COLUMNS).issubset(valid_processed.columns)

print("policy_train rows :", len(policy_train))
print("policy_test rows  :", len(policy_test))
print("holdout train rows:", len(train_processed))
print("holdout valid rows:", len(valid_processed))
print("feature count     :", len(HOLDOUT_FEATURE_COLUMNS))
print("categorical count :", len(HOLDOUT_CATEGORICAL_FEATURES))
print("split meta        :", split_meta)

In [ ]:
from catboost import CatBoostClassifier, CatBoostRegressor, Pool

@dataclass
class RetentionConfig:
    enabled: bool = False
    base_retention: float = 0.90
    elasticity: float = 4.0
    center: float = 0.0
    floor: float = 0.05
    cap: float = 0.995

@dataclass
class PricingEvaluation:
    lr_total: float
    lr_group1: float
    lr_group2: float
    share_group1: float
    violations: int
    score: float
    in_target: bool
    distance_to_target: float
    target_band: tuple[float, float]
    retention_rate: float | None = None
    retention_enabled: bool = False

    def to_dict(self) -> dict[str, Any]:
        return {
            "lr_total": self.lr_total,
            "lr_group1": self.lr_group1,
            "lr_group2": self.lr_group2,
            "share_group1": self.share_group1,
            "violations": self.violations,
            "policy_score": self.score,
            "in_target": self.in_target,
            "distance_to_target": self.distance_to_target,
            "target_band": {"min": self.target_band[0], "max": self.target_band[1]},
            "retention_rate": self.retention_rate,
            "retention_enabled": self.retention_enabled,
        }

def gini_from_auc(auc: float) -> float:
    return 2.0 * auc - 1.0

def classification_metrics(y_true: np.ndarray, p_pred: np.ndarray) -> dict[str, Any]:
    y_true = np.asarray(y_true)
    p_pred = np.asarray(p_pred)
    metrics: dict[str, Any] = {}

    if len(np.unique(y_true)) < 2:
        metrics["auc"] = None
        metrics["gini"] = None
    else:
        auc = float(roc_auc_score(y_true, p_pred))
        metrics["auc"] = auc
        metrics["gini"] = float(gini_from_auc(auc))

    metrics["brier"] = float(np.mean((y_true - p_pred) ** 2))
    return metrics

def severity_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, Any]:
    if len(y_true) == 0:
        return {"rmse": None, "mae": None, "r2": None}
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2 = float(r2_score(y_true, y_pred)) if len(y_true) >= 2 else None
    return {"rmse": rmse, "mae": mae, "r2": r2}

def prepare_model_input(
    df: pd.DataFrame,
    feature_columns: list[str],
    categorical_feature_columns: list[str],
) -> tuple[pd.DataFrame, list[int]]:
    X = df[feature_columns].copy()
    for column in feature_columns:
        if column in categorical_feature_columns:
            X[column] = X[column].astype("string").fillna("missing").astype(str)
        else:
            X[column] = pd.to_numeric(X[column], errors="coerce").fillna(0.0)
    cat_indices = [X.columns.get_loc(column) for column in categorical_feature_columns if column in X.columns]
    return X, cat_indices

def _fit_with_gpu_fallback(model_class: Any, pool: Pool, params: dict[str, Any], use_gpu: bool) -> tuple[Any, dict[str, Any]]:
    runtime_params = dict(params)
    if use_gpu:
        runtime_params["task_type"] = "GPU"
        runtime_params["devices"] = "0"

    try:
        model = model_class(**runtime_params)
        model.fit(pool)
        return model, {"task_type": runtime_params.get("task_type", "CPU"), "gpu_fallback": False}
    except Exception as exc:
        if not use_gpu:
            raise
        LOGGER.warning("GPU fit failed, retrying on CPU. Reason: %s", exc)
        runtime_params.pop("task_type", None)
        runtime_params.pop("devices", None)
        model = model_class(**runtime_params)
        model.fit(pool)
        return model, {
            "task_type": "CPU",
            "gpu_fallback": True,
            "gpu_error": str(exc),
        }

def fit_frequency_severity_bundle(
    train_df: pd.DataFrame,
    params: dict[str, Any],
    feature_columns: list[str],
    categorical_feature_columns: list[str],
    use_gpu: bool,
) -> dict[str, Any]:
    X_train, cat_indices = prepare_model_input(train_df, feature_columns, categorical_feature_columns)
    y_freq = train_df[TARGET_CLAIM_COL].fillna(0).astype(int).values

    classifier_params = {
        "iterations": int(params["iterations"]),
        "learning_rate": float(params["learning_rate"]),
        "depth": int(params["depth"]),
        "l2_leaf_reg": float(params["l2_leaf_reg"]),
        "random_strength": float(params["random_strength"]),
        "bagging_temperature": float(params["bagging_temperature"]),
        "border_count": int(params["border_count"]),
        "loss_function": "Logloss",
        "eval_metric": "AUC",
        "random_seed": int(params.get("random_state", SEED)),
        "thread_count": int(params.get("thread_count", -1)),
        "verbose": False,
    }

    freq_pool = Pool(X_train, y_freq, cat_features=cat_indices)
    frequency_model, freq_runtime = _fit_with_gpu_fallback(
        CatBoostClassifier,
        freq_pool,
        classifier_params,
        use_gpu=bool(use_gpu),
    )

    positive_mask = train_df[TARGET_CLAIM_COL].fillna(0).astype(int) > 0
    y_amount = pd.to_numeric(train_df[TARGET_AMOUNT_COL], errors="coerce").fillna(0.0).clip(lower=0.0)

    severity_model = None
    severity_constant = float(y_amount.loc[positive_mask].mean()) if int(positive_mask.sum()) > 0 else 0.0
    severity_fitted = False
    sev_runtime: dict[str, Any] = {"task_type": None, "gpu_fallback": False}

    if int(positive_mask.sum()) >= 20:
        X_pos = X_train.loc[positive_mask].copy()
        y_pos = np.log1p(y_amount.loc[positive_mask].values)

        severity_loss_name = str(params.get("severity_loss_function", "RMSE")).strip().upper()
        if severity_loss_name == "TWEEDIE":
            variance_power = float(np.clip(float(params.get("tweedie_variance_power", 1.5)), 1.01, 1.99))
            loss_function = f"Tweedie:variance_power={variance_power}"
        else:
            loss_function = "RMSE"

        regressor_params = {
            "iterations": int(params.get("reg_iterations", params["iterations"])),
            "learning_rate": float(params.get("reg_learning_rate", params["learning_rate"])),
            "depth": int(params.get("reg_depth", params["depth"])),
            "l2_leaf_reg": float(params.get("reg_l2_leaf_reg", params["l2_leaf_reg"])),
            "random_strength": float(params.get("reg_random_strength", params["random_strength"])),
            "bagging_temperature": float(params.get("reg_bagging_temperature", params["bagging_temperature"])),
            "border_count": int(params.get("reg_border_count", params["border_count"])),
            "loss_function": loss_function,
            "random_seed": int(params.get("random_state", SEED)),
            "thread_count": int(params.get("thread_count", -1)),
            "verbose": False,
        }

        sev_pool = Pool(X_pos, y_pos, cat_features=cat_indices)
        severity_model, sev_runtime = _fit_with_gpu_fallback(
            CatBoostRegressor,
            sev_pool,
            regressor_params,
            use_gpu=bool(use_gpu),
        )
        severity_fitted = True

    return {
        "frequency_model": frequency_model,
        "severity_model": severity_model,
        "severity_constant": severity_constant,
        "severity_fitted": severity_fitted,
        "feature_columns": list(feature_columns),
        "categorical_feature_columns": list(categorical_feature_columns),
        "runtime": {"frequency": freq_runtime, "severity": sev_runtime},
    }

def predict_frequency_severity_bundle(bundle: dict[str, Any], df: pd.DataFrame) -> pd.DataFrame:
    X, cat_indices = prepare_model_input(
        df,
        feature_columns=bundle["feature_columns"],
        categorical_feature_columns=bundle["categorical_feature_columns"],
    )
    pool = Pool(X, cat_features=cat_indices)

    p_claim = bundle["frequency_model"].predict_proba(pool)[:, -1]
    if bundle["severity_fitted"] and bundle["severity_model"] is not None:
        sev_log = bundle["severity_model"].predict(pool)
        expected_severity = np.expm1(np.clip(sev_log, 0, 20))
    else:
        expected_severity = np.full(len(df), float(bundle["severity_constant"]))

    expected_loss = p_claim * expected_severity
    return pd.DataFrame(
        {
            "p_claim": p_claim,
            "expected_severity": expected_severity,
            "expected_loss": expected_loss,
        },
        index=df.index,
    )

def _safe_ratio(numerator: float, denominator: float) -> float:
    return float(numerator / denominator) if denominator > 0 else 0.0

def estimate_retention_probabilities(
    base_premium: pd.Series,
    new_premium: pd.Series,
    retention_config: RetentionConfig | None = None,
) -> pd.Series:
    config = retention_config or RetentionConfig()
    if not config.enabled:
        return pd.Series(np.ones(len(base_premium), dtype=float), index=base_premium.index)

    base = pd.to_numeric(base_premium, errors="coerce").fillna(0.0).clip(lower=0.0)
    repriced = pd.to_numeric(new_premium, errors="coerce").fillna(0.0).clip(lower=0.0)
    price_delta = np.where(base > 0, repriced / base - 1.0, 0.0)

    base_retention = float(np.clip(config.base_retention, 1e-6, 1.0 - 1e-6))
    logit_base = float(np.log(base_retention / (1.0 - base_retention)))
    logits = logit_base - float(config.elasticity) * (price_delta - float(config.center))
    logits = np.clip(logits, -35.0, 35.0)
    probs = 1.0 / (1.0 + np.exp(-logits))

    floor = float(np.clip(config.floor, 0.0, 1.0))
    cap = float(np.clip(config.cap, 0.0, 1.0))
    if floor > cap:
        floor, cap = cap, floor
    probs = np.clip(probs, floor, cap)
    return pd.Series(probs, index=base_premium.index)

def apply_pricing_policy(
    df: pd.DataFrame,
    expected_loss: pd.Series,
    alpha: float,
    beta: float = 1.0,
) -> pd.Series:
    base_premium = pd.to_numeric(df[PREMIUM_COL], errors="coerce").fillna(0.0).clip(lower=0.0)
    expected_loss = pd.to_numeric(expected_loss, errors="coerce").fillna(0.0).clip(lower=0.0)

    mean_loss = float(expected_loss.mean()) if len(expected_loss) else 0.0
    if mean_loss <= 0:
        risk_index = np.ones(len(expected_loss))
    else:
        risk_index = expected_loss / mean_loss

    multiplier = 1.0 + float(alpha) * (risk_index - 1.0)
    raw_premium = base_premium * float(beta) * multiplier
    clipped = np.clip(raw_premium, np.zeros(len(base_premium)), 3.0 * base_premium)
    return pd.Series(clipped, index=df.index, name="new_premium")

def evaluate_pricing(
    df: pd.DataFrame,
    new_premium: pd.Series,
    target_lr: float,
    target_band: tuple[float, float] | None = None,
    retention_config: RetentionConfig | None = None,
) -> PricingEvaluation:
    base_premium = pd.to_numeric(df[PREMIUM_COL], errors="coerce").fillna(0.0).clip(lower=0.0)
    premium_wo_term = pd.to_numeric(df[PREMIUM_NET_COL], errors="coerce").fillna(base_premium)
    payout = pd.to_numeric(df[TARGET_AMOUNT_COL], errors="coerce").fillna(0.0).clip(lower=0.0)

    band_min, band_max = target_band if target_band is not None else (target_lr - 0.01, target_lr + 0.01)
    if band_min > band_max:
        band_min, band_max = band_max, band_min

    retention = estimate_retention_probabilities(base_premium, new_premium, retention_config=retention_config)
    retention_enabled = bool(retention_config.enabled) if retention_config is not None else False

    term_ratio = np.where(base_premium > 0, premium_wo_term / base_premium, 1.0)
    term_ratio = np.nan_to_num(term_ratio, nan=1.0, posinf=1.0, neginf=1.0).clip(min=0.0, max=1.0)
    new_premium_net = pd.Series(new_premium.values * term_ratio, index=df.index)

    premium_weight = retention if retention_enabled else pd.Series(np.ones(len(df), dtype=float), index=df.index)
    payout_weight = retention if retention_enabled else pd.Series(np.ones(len(df), dtype=float), index=df.index)

    group1_mask = new_premium <= base_premium
    group2_mask = ~group1_mask

    payout_total = float((payout * payout_weight).sum())
    den_total = float((new_premium_net * premium_weight).sum())
    lr_total = _safe_ratio(payout_total, den_total)

    payout_g1 = float((payout[group1_mask] * payout_weight[group1_mask]).sum())
    den_g1 = float((new_premium_net[group1_mask] * premium_weight[group1_mask]).sum())
    lr_group1 = _safe_ratio(payout_g1, den_g1)

    payout_g2 = float((payout[group2_mask] * payout_weight[group2_mask]).sum())
    den_g2 = float((new_premium_net[group2_mask] * premium_weight[group2_mask]).sum())
    lr_group2 = _safe_ratio(payout_g2, den_g2)

    share_group1 = _safe_ratio(float(premium_weight[group1_mask].sum()), float(premium_weight.sum())) if retention_enabled else float(group1_mask.mean())

    violations = int((new_premium < 0).sum() + (new_premium > 3.0 * base_premium).sum())
    distance_to_target = abs(lr_total - target_lr)
    in_target = band_min <= lr_total <= band_max

    score = (
        -distance_to_target
        - 0.7 * abs(lr_group1 - target_lr)
        - 0.7 * abs(lr_group2 - target_lr)
        + 0.15 * share_group1
        - 2.0 * violations
    )
    if in_target:
        score += 0.5

    retention_rate = float(retention.mean()) if retention_enabled else None
    if retention_rate is not None:
        score += 0.10 * retention_rate

    return PricingEvaluation(
        lr_total=lr_total,
        lr_group1=lr_group1,
        lr_group2=lr_group2,
        share_group1=share_group1,
        violations=violations,
        score=float(score),
        in_target=bool(in_target),
        distance_to_target=float(distance_to_target),
        target_band=(float(band_min), float(band_max)),
        retention_rate=retention_rate,
        retention_enabled=retention_enabled,
    )

def select_best_pricing(
    df: pd.DataFrame,
    expected_loss: pd.Series,
    target_lr: float,
    target_band: tuple[float, float],
    alpha_grid: np.ndarray = ALPHA_GRID,
    beta_grid: np.ndarray = BETA_GRID,
) -> tuple[float, float, pd.Series, PricingEvaluation]:
    best_alpha = float(alpha_grid[0])
    best_beta = float(beta_grid[0])
    best_premium = None
    best_eval = None

    for beta in beta_grid:
        for alpha in alpha_grid:
            premium = apply_pricing_policy(df=df, expected_loss=expected_loss, alpha=float(alpha), beta=float(beta))
            current_eval = evaluate_pricing(
                df=df,
                new_premium=premium,
                target_lr=target_lr,
                target_band=target_band,
            )
            rank_key = (
                1 if current_eval.violations == 0 else 0,
                1 if current_eval.in_target else 0,
                float(current_eval.score),
                float(current_eval.share_group1),
            )
            best_key = None if best_eval is None else (
                1 if best_eval.violations == 0 else 0,
                1 if best_eval.in_target else 0,
                float(best_eval.score),
                float(best_eval.share_group1),
            )
            if best_eval is None or rank_key > best_key:
                best_alpha = float(alpha)
                best_beta = float(beta)
                best_premium = premium
                best_eval = current_eval

    assert best_premium is not None
    assert best_eval is not None
    return best_alpha, best_beta, best_premium, best_eval

def run_holdout_experiment(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    params: dict[str, Any],
    feature_columns: list[str],
    categorical_feature_columns: list[str],
    use_gpu: bool,
    target_lr: float,
    target_band: tuple[float, float],
) -> dict[str, Any]:
    bundle = fit_frequency_severity_bundle(
        train_df=train_df,
        params=params,
        feature_columns=feature_columns,
        categorical_feature_columns=categorical_feature_columns,
        use_gpu=use_gpu,
    )
    valid_pred = predict_frequency_severity_bundle(bundle, valid_df)

    y_valid = valid_df[TARGET_CLAIM_COL].fillna(0).astype(int).values
    ml = classification_metrics(y_valid, valid_pred["p_claim"].values)

    pos_mask = valid_df[TARGET_CLAIM_COL].fillna(0).astype(int) > 0
    sev = severity_metrics(
        valid_df.loc[pos_mask, TARGET_AMOUNT_COL].fillna(0.0).values,
        valid_pred.loc[pos_mask, "expected_severity"].values,
    ) if int(pos_mask.sum()) > 0 else {"rmse": None, "mae": None, "r2": None}

    alpha, beta, new_premium, pricing_eval = select_best_pricing(
        df=valid_df,
        expected_loss=valid_pred["expected_loss"],
        target_lr=target_lr,
        target_band=target_band,
    )

    return {
        "bundle": bundle,
        "valid_pred": valid_pred,
        "ml": ml,
        "severity": sev,
        "alpha": float(alpha),
        "beta": float(beta),
        "new_premium": new_premium,
        "pricing": pricing_eval.to_dict(),
    }

INT_PARAM_KEYS = {
    "random_state",
    "thread_count",
    "iterations",
    "depth",
    "border_count",
    "reg_iterations",
    "reg_depth",
    "reg_border_count",
}

def cast_params_like_release(params: dict[str, Any]) -> dict[str, Any]:
    casted = dict(RELEASE_PARAMS)
    casted.update(params)
    for key in list(casted.keys()):
        if key in INT_PARAM_KEYS:
            casted[key] = int(casted[key])
        elif key != "severity_loss_function":
            casted[key] = float(casted[key]) if isinstance(casted[key], np.floating) else casted[key]
    casted["severity_loss_function"] = "TWEEDIE"
    return casted

def suggest_optuna_params(trial: optuna.Trial) -> dict[str, Any]:
    return {
        "iterations": trial.suggest_int("iterations", 180, 420),
        "learning_rate": trial.suggest_float("learning_rate", 0.015, 0.12, log=True),
        "depth": trial.suggest_int("depth", 4, 8),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 12.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 2.0),
        "border_count": trial.suggest_int("border_count", 64, 255),
        "reg_iterations": trial.suggest_int("reg_iterations", 250, 650),
        "reg_learning_rate": trial.suggest_float("reg_learning_rate", 0.01, 0.08, log=True),
        "reg_depth": trial.suggest_int("reg_depth", 4, 8),
        "reg_l2_leaf_reg": trial.suggest_float("reg_l2_leaf_reg", 1.0, 12.0, log=True),
        "reg_random_strength": trial.suggest_float("reg_random_strength", 0.0, 2.0),
        "reg_bagging_temperature": trial.suggest_float("reg_bagging_temperature", 0.0, 2.0),
        "reg_border_count": trial.suggest_int("reg_border_count", 64, 255),
        "tweedie_variance_power": trial.suggest_float("tweedie_variance_power", 1.05, 1.75),
    }

def tune_catboost_params(
    train_df: pd.DataFrame,
    valid_df: pd.DataFrame,
    feature_columns: list[str],
    categorical_feature_columns: list[str],
    use_gpu: bool,
    target_lr: float,
    target_band: tuple[float, float],
    n_trials: int,
) -> tuple[dict[str, Any], optuna.Study, pd.DataFrame]:
    def objective(trial: optuna.Trial) -> float:
        params = cast_params_like_release(suggest_optuna_params(trial))
        result = run_holdout_experiment(
            train_df=train_df,
            valid_df=valid_df,
            params=params,
            feature_columns=feature_columns,
            categorical_feature_columns=categorical_feature_columns,
            use_gpu=use_gpu,
            target_lr=target_lr,
            target_band=target_band,
        )
        trial.set_user_attr("gini", result["ml"].get("gini"))
        trial.set_user_attr("auc", result["ml"].get("auc"))
        trial.set_user_attr("lr_total", result["pricing"].get("lr_total"))
        trial.set_user_attr("in_target", result["pricing"].get("in_target"))
        return float(result["pricing"]["policy_score"])

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=SEED),
    )
    study.optimize(objective, n_trials=int(n_trials), show_progress_bar=False)

    best_params = cast_params_like_release(study.best_trial.params)
    trial_rows: list[dict[str, Any]] = []
    for trial in study.trials:
        row = dict(trial.params)
        row["value"] = trial.value
        row["state"] = str(trial.state)
        row["gini"] = trial.user_attrs.get("gini")
        row["auc"] = trial.user_attrs.get("auc")
        row["lr_total"] = trial.user_attrs.get("lr_total")
        row["in_target"] = trial.user_attrs.get("in_target")
        trial_rows.append(row)

    trials_df = pd.DataFrame(trial_rows)
    return best_params, study, trials_df

In [ ]:
optuna_study = None
optuna_trials_df = None
selected_params = dict(RELEASE_PARAMS)

if RUN_MODE == "optuna":
    selected_params, optuna_study, optuna_trials_df = tune_catboost_params(
        train_df=train_processed,
        valid_df=valid_processed,
        feature_columns=HOLDOUT_FEATURE_COLUMNS,
        categorical_feature_columns=HOLDOUT_CATEGORICAL_FEATURES,
        use_gpu=bool(USE_GPU),
        target_lr=float(TARGET_LR),
        target_band=tuple(TARGET_BAND),
        n_trials=int(OPTUNA_TRIALS),
    )
    write_json(OUTPUT_DIR / "best_params.json", selected_params)
    optuna_trials_df.to_csv(OUTPUT_DIR / "optuna_trials.csv", index=False)
    print("Optuna best value:", float(optuna_study.best_value))
else:
    print("Using release CatBoost parameters without Optuna.")

print(json.dumps(selected_params, ensure_ascii=False, indent=2))

In [ ]:
holdout_result = run_holdout_experiment(
    train_df=train_processed,
    valid_df=valid_processed,
    params=selected_params,
    feature_columns=HOLDOUT_FEATURE_COLUMNS,
    categorical_feature_columns=HOLDOUT_CATEGORICAL_FEATURES,
    use_gpu=bool(USE_GPU),
    target_lr=float(TARGET_LR),
    target_band=tuple(TARGET_BAND),
)

valid_policy_predictions = pd.DataFrame({
    CONTRACT_COL: valid_processed[CONTRACT_COL].values,
    PREMIUM_COL: pd.to_numeric(valid_processed[PREMIUM_COL], errors="coerce").fillna(0.0).values,
    PREMIUM_NET_COL: pd.to_numeric(valid_processed[PREMIUM_NET_COL], errors="coerce").fillna(0.0).values,
    TARGET_CLAIM_COL: pd.to_numeric(valid_processed[TARGET_CLAIM_COL], errors="coerce").fillna(0).astype(int).values,
    TARGET_AMOUNT_COL: pd.to_numeric(valid_processed[TARGET_AMOUNT_COL], errors="coerce").fillna(0.0).values,
    "p_claim": holdout_result["valid_pred"]["p_claim"].values,
    "expected_severity": holdout_result["valid_pred"]["expected_severity"].values,
    "expected_loss": holdout_result["valid_pred"]["expected_loss"].values,
    "new_premium": holdout_result["new_premium"].values,
})
valid_policy_predictions.to_csv(OUTPUT_DIR / "valid_policy_predictions.csv", index=False)

model_summary_df = pd.DataFrame([
    {
        "run_id": RUN_ID,
        "run_mode": RUN_MODE,
        "feature_count": len(HOLDOUT_FEATURE_COLUMNS),
        "categorical_feature_count": len(HOLDOUT_CATEGORICAL_FEATURES),
        "train_rows": len(train_processed),
        "valid_rows": len(valid_processed),
        "auc": holdout_result["ml"].get("auc"),
        "gini": holdout_result["ml"].get("gini"),
        "brier": holdout_result["ml"].get("brier"),
        "severity_rmse": holdout_result["severity"].get("rmse"),
        "severity_mae": holdout_result["severity"].get("mae"),
        "severity_r2": holdout_result["severity"].get("r2"),
        "alpha": holdout_result["alpha"],
        "beta": holdout_result["beta"],
        "lr_total": holdout_result["pricing"].get("lr_total"),
        "lr_group1": holdout_result["pricing"].get("lr_group1"),
        "lr_group2": holdout_result["pricing"].get("lr_group2"),
        "share_group1": holdout_result["pricing"].get("share_group1"),
        "policy_score": holdout_result["pricing"].get("policy_score"),
        "in_target": holdout_result["pricing"].get("in_target"),
        "distance_to_target": holdout_result["pricing"].get("distance_to_target"),
    }
])
model_summary_df.to_csv(OUTPUT_DIR / "model_summary.csv", index=False)

print(model_summary_df.T)

In [ ]:
full_preprocessor = fit_preprocessor(policy_train, PREPROCESSING_CONFIG)
train_full_processed = transform_with_preprocessor(policy_train, full_preprocessor)
test_policy_processed = transform_with_preprocessor(policy_test, full_preprocessor)

train_full_oof_te = build_oof_target_encoding_features(
    df=policy_train,
    state=full_preprocessor,
    target_column=TARGET_CLAIM_COL,
    n_splits=5,
    random_state=SEED,
    group_column=None,
)
for column in train_full_oof_te.columns:
    train_full_processed[column] = train_full_oof_te[column].values

FULL_FEATURE_COLUMNS = list(full_preprocessor.feature_columns)
FULL_CATEGORICAL_FEATURES = [
    column for column in full_preprocessor.categorical_output_columns if column in FULL_FEATURE_COLUMNS
]

assert set(FULL_FEATURE_COLUMNS).issubset(train_full_processed.columns)
assert set(FULL_FEATURE_COLUMNS).issubset(test_policy_processed.columns)

final_bundle = fit_frequency_severity_bundle(
    train_df=train_full_processed,
    params=selected_params,
    feature_columns=FULL_FEATURE_COLUMNS,
    categorical_feature_columns=FULL_CATEGORICAL_FEATURES,
    use_gpu=bool(USE_GPU),
)

test_policy_pred = predict_frequency_severity_bundle(final_bundle, test_policy_processed)
test_new_premium = apply_pricing_policy(
    df=test_policy_processed,
    expected_loss=test_policy_pred["expected_loss"],
    alpha=float(holdout_result["alpha"]),
    beta=float(holdout_result["beta"]),
)

test_policy_predictions = pd.DataFrame({
    CONTRACT_COL: test_policy_processed[CONTRACT_COL].values,
    PREMIUM_COL: pd.to_numeric(test_policy_processed[PREMIUM_COL], errors="coerce").fillna(0.0).values,
    PREMIUM_NET_COL: pd.to_numeric(test_policy_processed[PREMIUM_NET_COL], errors="coerce").fillna(0.0).values,
    "p_claim": test_policy_pred["p_claim"].values,
    "expected_severity": test_policy_pred["expected_severity"].values,
    "expected_loss": test_policy_pred["expected_loss"].values,
    "new_premium": test_new_premium.values,
})
test_policy_predictions.to_csv(OUTPUT_DIR / "test_policy_predictions.csv", index=False)

assert bool((test_policy_predictions["new_premium"] >= 0.0).all())
assert bool(
    (
        test_policy_predictions["new_premium"]
        <= 3.0 * test_policy_predictions[PREMIUM_COL].clip(lower=0.0)
    ).all()
)

submission_base_cols = [column for column in ["unique_id", CONTRACT_COL] if column in test_raw.columns]
submission = test_raw[submission_base_cols].copy()
submission = submission.merge(
    test_policy_predictions[[CONTRACT_COL, "new_premium"]],
    on=CONTRACT_COL,
    how="left",
)

if submission["new_premium"].isna().any():
    raise ValueError("submission contains NaN in new_premium after joining policy predictions to raw test rows")

assert len(submission) == len(test_raw)
submission.to_csv(OUTPUT_DIR / "submission.csv", index=False)

preprocess_metadata = {
    "run_id": RUN_ID,
    "holdout_split": split_meta,
    "holdout": {
        "feature_count": len(HOLDOUT_FEATURE_COLUMNS),
        "categorical_feature_count": len(HOLDOUT_CATEGORICAL_FEATURES),
        "feature_columns": HOLDOUT_FEATURE_COLUMNS,
        "categorical_feature_columns": HOLDOUT_CATEGORICAL_FEATURES,
        "preprocessor": holdout_preprocessor.to_dict(),
    },
    "full_train": {
        "feature_count": len(FULL_FEATURE_COLUMNS),
        "categorical_feature_count": len(FULL_CATEGORICAL_FEATURES),
        "feature_columns": FULL_FEATURE_COLUMNS,
        "categorical_feature_columns": FULL_CATEGORICAL_FEATURES,
        "preprocessor": full_preprocessor.to_dict(),
    },
}
write_json(OUTPUT_DIR / "preprocess_metadata.json", preprocess_metadata)

artifacts_payload = {
    "valid_policy_predictions": str(OUTPUT_DIR / "valid_policy_predictions.csv"),
    "test_policy_predictions": str(OUTPUT_DIR / "test_policy_predictions.csv"),
    "model_summary": str(OUTPUT_DIR / "model_summary.csv"),
    "submission": str(OUTPUT_DIR / "submission.csv"),
    "preprocess_metadata": str(OUTPUT_DIR / "preprocess_metadata.json"),
}
if RUN_MODE == "optuna":
    artifacts_payload["best_params"] = str(OUTPUT_DIR / "best_params.json")
    artifacts_payload["optuna_trials"] = str(OUTPUT_DIR / "optuna_trials.csv")

metrics_payload = {
    "run_id": RUN_ID,
    "run_mode": RUN_MODE,
    "selected_params": selected_params,
    "config": {
        "train_csv": str(TRAIN_CSV),
        "test_csv": str(TEST_CSV),
        "output_dir": str(OUTPUT_DIR),
        "use_gpu": bool(USE_GPU),
        "sample_n": SAMPLE_N,
        "time_holdout_start": TIME_HOLDOUT_START,
        "target_lr": float(TARGET_LR),
        "target_band": {"min": float(TARGET_BAND[0]), "max": float(TARGET_BAND[1])},
        "optuna_trials": int(OPTUNA_TRIALS),
    },
    "preprocessing": {
        "policy_train_rows": int(len(policy_train)),
        "policy_test_rows": int(len(policy_test)),
        "holdout_train_rows": int(len(train_processed)),
        "holdout_valid_rows": int(len(valid_processed)),
        "holdout_feature_count": int(len(HOLDOUT_FEATURE_COLUMNS)),
        "full_train_feature_count": int(len(FULL_FEATURE_COLUMNS)),
    },
    "holdout": {
        "split_meta": split_meta,
        "ml": holdout_result["ml"],
        "severity": holdout_result["severity"],
        "pricing": holdout_result["pricing"],
        "alpha": float(holdout_result["alpha"]),
        "beta": float(holdout_result["beta"]),
        "runtime": holdout_result["bundle"]["runtime"],
    },
    "final_refit": {
        "runtime": final_bundle["runtime"],
        "submission_rows": int(len(submission)),
        "policy_test_rows": int(len(test_policy_predictions)),
    },
    "artifacts": artifacts_payload,
}
write_json(OUTPUT_DIR / "metrics.json", metrics_payload)

print("Saved artifacts:")
for key, value in artifacts_payload.items():
    print(f"  {key}: {value}")
print("  metrics:", OUTPUT_DIR / "metrics.json")
submission.head()

## Next steps

- For the fastest reproducible run, keep RUN_MODE = "release".
- For a longer GPU search, switch to RUN_MODE = "optuna" and increase OPTUNA_TRIALS.
- The final artifacts are written to a timestamped directory under OUTPUT_ROOT.